# Colab runtime and repository check

Select the Colab kernel in VS Code. Run this notebook on the hosted runtime before any model experiment. The extension's remote kernel does not automatically see local Windows files. Upload the repository folder to the server, or clone it there with your own access method. This notebook reads no private credentials and does not train anything.


In [ ]:
from pathlib import Path
import sys
print("Python:", sys.version.split()[0])
print("Working directory:", Path.cwd())
try:
    import torch
    print("Torch:",torch.__version__)
    mps = bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())
    cuda = torch.cuda.is_available()
    print("MPS available:", mps)
    print("CUDA available:", cuda)
    device = torch.device("mps" if mps else "cuda" if cuda else "cpu")
    print("Selected device:", device)
    if cuda and device.type == "cuda":
        print("GPU:",torch.cuda.get_device_name(0))
    if device.type != "cpu":
        x = torch.ones((8, 8), device=device, requires_grad=True)
        (x @ x).sum().backward()
        print("Device forward/backward: OK")
except ImportError:
    print("PyTorch not installed")


In [ ]:
root = next((p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/CLIP-Animal-Classifier")] if (p / "notebooks/paper_test/verification/verify_paper.py").exists()), None)
assert root is not None, "Upload the repository to the Colab runtime; expected /content/CLIP-Animal-Classifier."
required = ["paper/paper_CLIP_Gemma4_revised_v2.md",
            "notebooks/paper_test/verification/verify_paper.py",
            "Semester/results/table7_full_benchmark_results.json",
            "Semester/results/verified_test_predictions.pt"]
for name in required:
    p=root/name
    print(("OK" if p.exists() else "MISSING"),name)
assert all((root/name).exists() for name in required), "Some inputs are absent on Colab."
print("Repository root:",root)


experiment_inputs = {
    "CuPL Oxford": [
        "notebooks/paper_test/external/pets_prompts_full.json",
        "Semester/cached_features/oxford_pets/test_clip.pt",
        "Semester/cached_features/oxford_pets/test_labels.pt",
        "dataset/cached/test_features_tta.pt",
    ],
    "Stanford DualStream": [
        "Semester/cached_features/stanford_dogs/train_clip.pt",
        "Semester/cached_features/stanford_dogs/train_labels.pt",
        "Semester/cached_features/stanford_dogs/test_clip.pt",
        "Semester/cached_features/stanford_dogs/test_labels.pt",
        "Semester/attributes/stanford_dogs_attributes.json",
    ],
}
for experiment, files in experiment_inputs.items():
    missing = [name for name in files if not (root / name).exists()]
    print(experiment, "inputs:", "OK" if not missing else "MISSING " + ", ".join(missing))
